# 13 — Solve and compare Direct/HB responses

> **Lesson focus**
>
> **Learn:** solve one selected network through Direct and pump-off HB.
> **Run:** materialize two independent response Results. **Inspect:**
> each Result’s S/Y/Z surface and named HB case. **Status:**
> `STABILIZED` Full V1.

## Solve one view on two declared grids

Direct and HB consume the same backend-neutral lineage. Their Specs
differ because HB also declares axes, drives, cases, and truncation;
their public selected-network S/Y/Z surfaces use the same selected-view
convention. Their frequency grids are nevertheless independently
declared request data: SCNSim does not manufacture a shared grid or
interpolate one Result onto the other. The HB request retains its
declared odd pump-drive mode through `four_wave_mixing=True`; its
`pump_off` case still supplies no drive current.

In [ ]:
from fixtures.floating_probe import build_floating_probe_circuit
from scnsim import (
    CircuitRun,
    CurrentDrive,
    DirectSolveSpec,
    HBCaseSpec,
    HBSolveSpec,
    HBTruncation,
    PumpAxis,
    ReductionPipeline,
    SParameterTrace,
    units as u,
)

fixture = build_floating_probe_circuit()
run = CircuitRun(plan=fixture.plan, workspace="workspaces/advanced-course")
view = run.original.reduce(
    ReductionPipeline()
    .ptc(fixture.probe_plus, fixture.probe_minus)
    .transform_pair(fixture.qubit_plus, fixture.qubit_minus, id="qubit")
    .retain("feedline_in", "feedline_out", "qubit.differential")
)
direct_frequencies = [5.5, 6.0, 6.5] * u.GHz
direct_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
direct = run.solve(
    view,
    DirectSolveSpec(frequencies=direct_frequencies, traces=(direct_trace,)),
)

pump = PumpAxis(id="pump", frequency=9.0 * u.GHz)
pump_drive = CurrentDrive(id="pump_drive", at=fixture.feedline_in, mode=(1,))
hb_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(0,),
    output_port="feedline_out",
    output_mode=(0,),
)
hb = run.solve(
    view,
    HBSolveSpec(
        pump_axes=(pump,),
        drives=(pump_drive,),
        frequencies=[5.4, 5.9, 6.4] * u.GHz,
        cases=(HBCaseSpec(id="pump_off", currents={}),),
        truncation=HBTruncation(
            pump_harmonics=(3,),
            modulation_harmonics=(1,),
            three_wave_mixing=False,
            four_wave_mixing=True,
        ),
        traces=(hb_trace,),
    ),
)

## Present the materialized Results side by side

Case lookup uses the user-declared ID. `magnitude` changes presentation
only; it does not recompute, clip exact zeros, interpolate either
result, or derive a Direct/HB residual. This lesson presents two
independently materialized Results, not a `compare()` operation.

In [ ]:
pump_off = hb.cases["pump_off"]
direct.traces["transmission"].show(magnitude="db")
pump_off.traces["transmission"].show(magnitude="db")
direct.s.view
pump_off.s.view
direct.y.view
pump_off.y.view
direct.z.view
pump_off.z.view

The two curves may share a figure only as separate, identity-bearing
traces. Any numerical comparison belongs to an explicit downstream
calculation on already materialized compatible quantities; V1 has no
interpolation, grid alignment, peak matching, or Direct/HB comparison
residual API.

[Previous](12_prepare_hb.qmd) · [Course map](../../docs/index.qmd) ·
[Concept: selected-network
symmetry](../../docs/concepts/direct-and-hb-realizations.qmd#direct-and-hb-selected-network)
· [Contract](../../docs/v1-runtime-contract.qmd)